In [1]:
#!pip install adversarial-robustness-toolbox

In [2]:
import numpy as np
import tensorflow as tf
import keras
from keras.datasets import cifar10
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from keras import backend as K
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification import KerasClassifier


In [3]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Create a neural network model
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=x_train.shape[1:]))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))

In [4]:
# Compile and train the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [5]:
# reshape data to fit model
X_train, X_test = x_train/255, x_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# Re-train the model on the augmented dataset
model.fit(X_train, y_train, batch_size=32, epochs=5, validation_data=(X_test, y_test))
# Evaluate the model's accuracy on the test dataset
score1 = model.evaluate(X_test, y_test, verbose=0)
print('Test accuracy:', score1[1])

Epoch 1/5
1563/1563 [==============================] - 73s 42ms/step - loss: 1.4578 - accuracy: 0.4745 - val_loss: 1.2370 - val_accuracy: 0.5624
Epoch 2/5
1563/1563 [==============================] - 45s 29ms/step - loss: 1.0919 - accuracy: 0.6184 - val_loss: 1.0968 - val_accuracy: 0.6193
Epoch 3/5
1563/1563 [==============================] - 44s 28ms/step - loss: 0.9563 - accuracy: 0.6683 - val_loss: 0.9823 - val_accuracy: 0.6615
Epoch 4/5
1563/1563 [==============================] - 43s 27ms/step - loss: 0.8692 - accuracy: 0.6975 - val_loss: 0.9513 - val_accuracy: 0.6715
Epoch 5/5
1563/1563 [==============================] - 43s 28ms/step - loss: 0.7944 - accuracy: 0.7227 - val_loss: 0.9733 - val_accuracy: 0.6728
Test accuracy: 0.6728000044822693


In [ ]:
# Create a KerasClassifier instance
tf.compat.v1.disable_eager_execution()
classifier = KerasClassifier(model=model, clip_values=(0, 1), use_logits=False)

In [ ]:
# Generate adversarial examples
jsma = SaliencyMapMethod(classifier, theta=1, gamma=0.1)
x_train_adv = jsma.generate(X_train)

In [ ]:
x_train_adv = np.concatenate((X_train, x_train_adv))
y_train_adv = np.concatenate((y_train, y_train))

In [ ]:
model.fit(x_train_adv, y_train_adv, batch_size=32, epochs=5, validation_data=(X_test, y_test))

# Evaluate the model's accuracy on the test dataset
score = model.evaluate(X_test, y_test, verbose=0)
print('Test accuracy:', score[1])